# COMPASS multivariate_longitudinal models

Dynamic-DeepHit and SurvLatent ODE, each in `platinum` (death censored,
comparable to the Cox/XGBoost arms) and `competing` (platinum + death as
competing causes) configurations, at every landmark. Requires
`01_preprocessing.ipynb` to have built the merged `profile_data` inputs
under `prediction_inputs_<arm>/` with `--build-longitudinal` (the default).

Kept separate from `03_multivariate.ipynb` so that notebook remains runnable
in a torch-free environment. This one requires torch (Dynamic-DeepHit) and,
for SurvLatent ODE, a cloned `itmoon7/survlatent_ode` repo with its own conda
env active -- see `multivariate_longitudinal/README.md`.

In [ ]:
ARMS = ["adt"]

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.MAX_PRED_WINDOW = 260

# Path to a cloned itmoon7/survlatent_ode repo (with its conda env active).
# Required only for the survlatent-ode tasks -- build_model_command raises a
# clear RuntimeError if it is still None when one is dispatched.
cp.SURVLATENT_REPO = None  # e.g. "/data/gusev/USERS/jpconnor/repos/survlatent_ode"

RUNS = cp.make_runs(ARMS)

## Run multivariate_longitudinal models

Dynamic-DeepHit (platinum/competing) and SurvLatent ODE (platinum/competing)
arms. Set `cp.FORCE_RERUN = False` to skip tasks whose metrics file already
exists. Layout:
`local_runs_<arm>/multivariate_longitudinal/<model>/landmark_{0,90,180}/<platinum,competing>/`.

In [ ]:
for run in RUNS:
    cp.run_multivariate_longitudinal(run)

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), filtered to the platinum row for headline comparability against
`03_multivariate.ipynb`'s Cox/XGBoost arms, then combined across runs.

In [ ]:
summary_dfs = {run["label"]: cp.summarize_longitudinal_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_longitudinal_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_longitudinal_summary_df